In [ ]:
!pip install -q transformers[torch] datasets sentencepiece accelerate evaluate sacrebleu rouge_score


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 2.6 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
from google.colab import drive

def connect_google_drive(mount_path='/content/drive'):
    drive.mount(mount_path)
connect_google_drive()

Mounted at /content/drive


In [ ]:
from IPython.display import display

def inspect_dataset(file_path):
    if not os.path.exists(file_path):
        print(f"KHÔNG TÌM THẤY FILE: {file_path}")
        print("Vui lòng kiểm tra lại đường dẫn file trên Drive của bạn!")
        return None

    ext = os.path.splitext(file_path)[-1].lower()
    try:
        if ext == '.csv':
            df = pd.read_csv(file_path)
        elif ext in ['.xlsx', '.xls']:
            df = pd.read_excel(file_path)
        elif ext == '.json':
            df = pd.read_json(file_path)
        elif ext == '.parquet':
            df = pd.read_parquet(file_path)
        elif ext == '.txt':
            df = pd.read_csv(file_path, sep='\t', header=None, names=['text'])
        else:
            print(f"Định dạng {ext} chưa được hỗ trợ tự động.")
            return None
    except Exception as e:
        print(f"Lỗi khi đọc file: {e}")
        return None

    # Tổng quan kích thước
    print(f"TỔNG QUAN:")
    print(f"   - Tổng số dòng (Rows):    {len(df):,}")
    print(f"   - Tổng số cột (Columns): {len(df.columns)}")
    print(f"   - Tên các cột:           {list(df.columns)}\n")

    #Kiểm tra ô trống (Null/Missing values)
    print("KIỂM TRA GIÁ TRỊ TRỐNG (NULL / MISSING):")
    missing_df = pd.DataFrame({
        'Kiểu dữ liệu': df.dtypes,
        'Số ô trống': df.isnull().sum(),
        'Tỷ lệ ô trống (%)': (df.isnull().sum() / len(df) * 100).round(2)
    })
    print(missing_df.to_string())
    print()

    #Xem 5 dòng đầu tiên
    print("XEM 5 DÒNG ĐẦU TIÊN (HEAD):")
    display(df.head())
    print()

    #Xem 5 mẫu ngẫu nhiên
    print("XEM 5 MẪU NGẪU NHIÊN (RANDOM SAMPLE):")
    display(df.sample(min(5, len(df)), random_state=42))
    print()

    #Phân tích độ dài câu (đối với văn bản)
    print("THỐNG KÊ ĐỘ DÀI KÝ TỰ CỦA CÁC CỘT VĂN BẢN:")
    for col in df.columns:
        if df[col].dtype == 'object':
            lengths = df[col].astype(str).str.len()
            print(f"   Cột [{col}]:")
            print(f"     + Ngắn nhất:  {lengths.min()} ký tự")
            print(f"     + Trung bình:  {lengths.mean():.1f} ký tự")
            print(f"     + Dài nhất:   {lengths.max()} ký tự")

    return df

FOLDER_PATH = "/content/drive/MyDrive/Xử Lý Ngôn Ngữ Tự Nhiên/Tiểu Luận/Dataset/Data-vi-ko"

# Đường dẫn chính xác tới 2 file .txt
vi_file = os.path.join(FOLDER_PATH, "TED2020vi.txt")
ko_file = os.path.join(FOLDER_PATH, "TED2020kor.txt")

#Đọc nội dung 2 file
with open(vi_file, "r", encoding="utf-8") as f:
    vi_lines = [line.strip() for line in f]

with open(ko_file, "r", encoding="utf-8") as f:
    ko_lines = [line.strip() for line in f]

#Ghép thành DataFrame 2 cột Việt - Hàn
df = pd.DataFrame({
    'vi': vi_lines,
    'ko': ko_lines
})

#Lưu thành file CSV duy nhất vào cùng thư mục
SAVE_CSV_PATH = os.path.join(FOLDER_PATH, "vi_ko_dataset.csv")
df.to_csv(SAVE_CSV_PATH, index=False, encoding="utf-8-sig")

#Gọi hàm inspect_dataset trên file CSV vừa tạo
df_inspected = inspect_dataset(SAVE_CSV_PATH)

TỔNG QUAN:
   - Tổng số dòng (Rows):    323,070
   - Tổng số cột (Columns): 2
   - Tên các cột:           ['vi', 'ko']

KIỂM TRA GIÁ TRỊ TRỐNG (NULL / MISSING):
   Kiểu dữ liệu  Số ô trống  Tỷ lệ ô trống (%)
vi       object        3087               0.96
ko       object        3087               0.96

XEM 5 DÒNG ĐẦU TIÊN (HEAD):


,vi,ko
0,"Cám ơn rất nhiều, Chris.","감사합니다, 크리스."
1,Đây thật sự là một vinh hạnh lớn cho tôi khi c...,이곳에 두 번이나 설 기회를 얻게된 것을 큰 영광으로 생각합니다. 진심으로 감사드립니다.
2,"Tôi thực sự bị choáng ngợp bởi hội nghị này, v...",저는 TED에 큰 감명을 받았습니다. 그리고 제가 지난 번에 한 이야기에 대해 많은...
3,"Tôi nói điều này thật lòng, một phần bởi vì (g...",그리고 진심으로 하는 말인데...(흑) 제겐 관심이 좀 필요했어요!
4,(Cười) Hãy thử đặt mình vào vị trí của tôi xem.,여러분도 제 입장이 되보세요!



XEM 5 MẪU NGẪU NHIÊN (RANDOM SAMPLE):


,vi,ko
146797,Không thể nói trước điều gì.,아무도 예측할 수 없어요.
218204,"Những bác sĩ chăm sóc Paul, làm tôi cảm thông ...",폴을 돌보는 임상의는 제 동료들에게 더 깊은 감사를 전했습니다.
274684,"Tính đến giờ, tôi đã dành ra mười năm cho việc...",저는 10년 동안 전문적인 교육을 받았는데 5년째 그것들을 버리는 중입니다.
155197,"Tôi là một thổ địa của Lagos và Berlin,"" cho t...","""저는 라고스와 베를린 사람입니다. ""는 부정되거나 없앨 수 없는 여러 겹의 경험이..."
198363,Họ nhìn thấy một chiếc xe đạp trẻ em không khó...,그러다 어느 집 현관에 있던 자전거와 스쿠터를 발견하고는 어리석게도 그걸 집어 탔어요.



THỐNG KÊ ĐỘ DÀI KÝ TỰ CỦA CÁC CỘT VĂN BẢN:
   Cột [vi]:
     + Ngắn nhất:  1 ký tự
     + Trung bình:  95.0 ký tự
     + Dài nhất:   3312 ký tự
   Cột [ko]:
     + Ngắn nhất:  1 ký tự
     + Trung bình:  48.7 ký tự
     + Dài nhất:   1825 ký tự


In [ ]:
import unicodedata
from sklearn.model_selection import train_test_split

FOLDER_PATH = "/content/drive/MyDrive/Xử Lý Ngôn Ngữ Tự Nhiên/Tiểu Luận/Dataset/Data-vi-ko"
CSV_PATH = os.path.join(FOLDER_PATH, "vi_ko_dataset.csv")

print("Đã khai báo xong thư viện và đường dẫn!")

Đã khai báo xong thư viện và đường dẫn!


In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Số lượng ban đầu: {len(df):,} cặp câu")

#Loại bỏ dòng ô trống (Null)
df = df.dropna().reset_index(drop=True)

#Chuẩn hóa Unicode NFC cho tiếng Việt và xóa khoảng trắng thừa
def clean_text(text):
    text = unicodedata.normalize('NFC', str(text))
    return text.strip()

df['vi'] = df['vi'].apply(clean_text)
df['ko'] = df['ko'].apply(clean_text)

#Loại bỏ các cặp câu trùng lặp
df = df.drop_duplicates(subset=['vi', 'ko']).reset_index(drop=True)

#Lọc độ dài câu (loại bỏ câu quá ngắn < 2 ký tự hoặc quá dài > 250 ký tự)
MIN_LEN, MAX_LEN = 2, 250
df = df[
    (df['vi'].str.len() >= MIN_LEN) & (df['vi'].str.len() <= MAX_LEN) &
    (df['ko'].str.len() >= MIN_LEN) & (df['ko'].str.len() <= MAX_LEN)
].reset_index(drop=True)

print(f"Số lượng sau khi làm sạch: {len(df):,} cặp câu")

Số lượng ban đầu: 323,070 cặp câu
Số lượng sau khi làm sạch: 306,125 cặp câu


In [ ]:
from sklearn.model_selection import train_test_split

# 1. Chia tập dữ liệu df thành Train (80%), Val (10%), Test (10%)
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# 2. Lưu lại các file CSV
train_df.to_csv(os.path.join(FOLDER_PATH, "train_split.csv"), index=False, encoding="utf-8-sig")
val_df.to_csv(os.path.join(FOLDER_PATH, "val_split.csv"), index=False, encoding="utf-8-sig")
test_df.to_csv(os.path.join(FOLDER_PATH, "test_split.csv"), index=False, encoding="utf-8-sig")

print(f"chia dữ liệu cho 2 chiều :")
print(f"Tập Train: {len(train_df)} mẫu")
print(f"Tập Val  : {len(val_df)} mẫu")
print(f"Tập Test : {len(test_df)} mẫu")

chia dữ liệu cho 2 chiều :
Tập Train: 244900 mẫu
Tập Val  : 30612 mẫu
Tập Test : 30613 mẫu


In [ ]:
from sklearn.model_selection import train_test_split

# Chia 80% Train từ df (chứa song song 2 cột 'vi' và 'ko')
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

# Chia 20% temp thành 10% Validation và 10% Test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42
)

print("KẾT QUẢ CHIA TẬP DỮ LIỆU (CHO 2 MÔ HÌNH RIÊNG):")
print(f"   - Tập Train:       {len(train_df):,} mẫu (80%)")
print(f"   - Tập Validation: {len(val_df):,} mẫu (10%)")
print(f"   - Tập Test:        {len(test_df):,} mẫu (10%)")

KẾT QUẢ CHIA TẬP DỮ LIỆU (CHO 2 MÔ HÌNH RIÊNG):
   - Tập Train:       244,900 mẫu (80%)
   - Tập Validation: 30,612 mẫu (10%)
   - Tập Test:        30,613 mẫu (10%)


In [ ]:
train_df.to_csv(os.path.join(FOLDER_PATH, "train.csv"), index=False, encoding="utf-8-sig")
val_df.to_csv(os.path.join(FOLDER_PATH, "val.csv"), index=False, encoding="utf-8-sig")
test_df.to_csv(os.path.join(FOLDER_PATH, "test.csv"), index=False, encoding="utf-8-sig")

print("Đã lưu thành công 3 file: train.csv, val.csv, test.csv vào Drive!")

Đã lưu thành công 3 file: train.csv, val.csv, test.csv vào Drive!


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

FOLDER_PATH = "/content/drive/MyDrive/Xử Lý Ngôn Ngữ Tự Nhiên/Tiểu Luận/Dataset/Data-vi-ko"
MODEL_NAME = "facebook/nllb-200-distilled-600M"

# Kiểm tra GPU
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

# Tải Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Đã tải thành công Tokenizer NLLB-200!")

CUDA Available: True
Device Name: Tesla T4


Đã tải thành công Tokenizer NLLB-200!


In [ ]:
import os
import gc
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "facebook/nllb-200-distilled-600M"

# 1. Nạp df_test nếu chưa có
if 'df_test' not in globals():
    FOLDER_PATH = "/content/drive/MyDrive/Xử Lý Ngôn Ngữ Tự Nhiên/Tiểu Luận/Dataset/Data-vi-ko"
    TEST_CSV_PATH = os.path.join(FOLDER_PATH, "test.csv")
    df_test = pd.read_csv(TEST_CSV_PATH)

# 2. Lấy ngẫu nhiên câu mẫu
sample_row = df_test.sample(1, random_state=42).iloc[0]
samples = [
    {"direction": "Chiều 1: Việt -> Hàn (vi2ko)", "text": sample_row['vi'], "src_lang": "vie_Latn"},
    {"direction": "Chiều 2: Hàn -> Việt (ko2vi)", "text": sample_row['ko'], "src_lang": "kor_Hang"}
]

# 3. Nạp đúng Tokenizer và Mô hình NLLB
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
eval_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

print("\nTOKENS & EMBEDDING (NLLB-200)")

for item in samples:
    label = item["direction"]
    text = item["text"]
    tokenizer.src_lang = item["src_lang"]

    tokens = tokenizer.tokenize(text)
    token_ids = tokenizer.convert_tokens_to_ids(tokens)
    decoded_text = tokenizer.decode(token_ids, skip_special_tokens=True)

    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        # Đã sửa: Dùng .get_encoder() cho mô hình NLLB/M2M100
        encoder_outputs = eval_model.get_encoder()(**inputs)
        token_embeddings = encoder_outputs.last_hidden_state
        sentence_embedding = token_embeddings.mean(dim=1)

    print(f"\n>>> {label.upper()}")
    print(f"Câu ngẫu nhiên         : \"{text}\"")
    print(f"Số lượng Tokens        : {len(tokens)}")
    print(f"Danh sách Tokens       : {tokens[:10]}...")
    print(f"Danh sách Token IDs    : {token_ids[:10]}...")
    print(f"Decode lại từ IDs      : \"{decoded_text}\"")
    print(f"Ma trận Token Vector   : {list(token_embeddings.shape)}")
    print(f"Vector câu (Mean Pool) : {list(sentence_embedding.shape)}")
    print(f"5 giá trị đầu vector   : {sentence_embedding[0][:5].cpu().numpy().round(4)}")
    print("-" * 75)

# 4. GIẢI PHÓNG VRAM GPU
del eval_model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print("Đã dọn dẹp VRAM GPU sạch sẽ, sẵn sàng cho bước Train!")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]


TOKENS & EMBEDDING (NLLB-200)

>>> CHIỀU 1: VIỆT -> HÀN (VI2KO)
Câu ngẫu nhiên         : "Điều này có nghĩa là họ là những người nhẫn tâm và hung hãn nhất trong tất cả các tù nhân."
Số lượng Tokens        : 23
Danh sách Tokens       : ['▁Điều', '▁này', '▁có', '▁nghĩa', '▁là', '▁họ', '▁là', '▁những', '▁người', '▁nhẫn']...
Danh sách Token IDs    : [31772, 4357, 1601, 26769, 1282, 4116, 1282, 3379, 3071, 215395]...
Decode lại từ IDs      : "Điều này có nghĩa là họ là những người nhẫn tâm và hung hãn nhất trong tất cả các tù nhân."
Ma trận Token Vector   : [1, 25, 1024]
Vector câu (Mean Pool) : [1, 1024]
5 giá trị đầu vector   : [ 0.1008 -0.006  -0.0061  0.3224  0.0192]
---------------------------------------------------------------------------

>>> CHIỀU 2: HÀN -> VIỆT (KO2VI)
Câu ngẫu nhiên         : "그 말은 제 연구 대상들이 재소자들 중에서도 가장 냉담하고 공격적이란 뜻이었습니다."
Số lượng Tokens        : 21
Danh sách Tokens       : ['▁그', '▁말은', '▁제', '▁연구', '▁대상', '들이', '▁재', '소', '자들', '▁중']...
Danh sách Token IDs  

In [ ]:
import os
import gc
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
MODEL_NAME = "facebook/nllb-200-distilled-600M"

def train_single_direction(train_df, val_df, src_col, tgt_col, model_save_path, checkpoint_path):
    gc.collect()
    torch.cuda.empty_cache()

    src_lang = "vie_Latn" if src_col == "vi" else "kor_Hang"
    tgt_lang = "kor_Hang" if src_col == "vi" else "vie_Latn"

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang=src_lang, tgt_lang=tgt_lang)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

    # Bật Gradient Checkpointing tiết kiệm VRAM
    model.gradient_checkpointing_enable()

    # Cắt lấy 40,000 mẫu train để tối ưu thời gian huấn luyện
    train_sub = train_df.sample(min(40000, len(train_df)), random_state=42).reset_index(drop=True)
    train_dataset = Dataset.from_pandas(train_sub[[src_col, tgt_col]])

    val_sub = val_df.sample(min(1000, len(val_df)), random_state=42).reset_index(drop=True)
    val_dataset = Dataset.from_pandas(val_sub[[src_col, tgt_col]])

    def preprocess_fn(examples):
        inputs = [str(ex) for ex in examples[src_col]]
        targets = [str(ex) for ex in examples[tgt_col]]

        model_inputs = tokenizer(inputs, max_length=128, truncation=True)
        labels = tokenizer(text_target=targets, max_length=128, truncation=True)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    tokenized_train = train_dataset.map(preprocess_fn, batched=True, remove_columns=[src_col, tgt_col])
    tokenized_val = val_dataset.map(preprocess_fn, batched=True, remove_columns=[src_col, tgt_col])

    training_args = Seq2SeqTrainingArguments(
        output_dir=checkpoint_path,
        per_device_train_batch_size=8,        # Tăng batch size lên 8 để đẩy nhanh tốc độ
        gradient_accumulation_steps=2,        # Effective Batch Size = 16
        per_device_eval_batch_size=8,
        learning_rate=5e-5,
        warmup_steps=300,
        num_train_epochs=2,                    # Giảm xuống 2 Epochs
        predict_with_generate=True,
        fp16=torch.cuda.is_available(),
        save_strategy="no",
        eval_strategy="no",
        logging_steps=100,
        report_to="none"
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        processing_class=tokenizer,
        data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    )

    print(f"\nBẮT ĐẦU TRAIN CHIỀU: {src_col.upper()} -> {tgt_col.upper()} ({len(train_sub):,} MẪU)")
    trainer.train()

    model.save_pretrained(model_save_path)
    tokenizer.save_pretrained(model_save_path)
    print(f"Đã lưu mô hình thành công tại: {model_save_path}")

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# Train Mô hình 1 (Việt -> Hàn)
PATH_VI2KO_MODEL = os.path.join(FOLDER_PATH, "nllb_vi2ko_model")
PATH_VI2KO_CKPT = os.path.join(FOLDER_PATH, "checkpoint_vi2ko")

# Kiểm tra xem mô hình đã được lưu đầy đủ trên Drive chưa
config_file = os.path.join(PATH_VI2KO_MODEL, "config.json")

if os.path.exists(config_file):
    print(f"Mô hình Việt -> Hàn ĐÃ TỒN TẠI tại: {PATH_VI2KO_MODEL}")
    print("-> Bỏ qua bước huấn luyện và chuyển sang cell tiếp theo!")
else:
    print(f"Chưa tìm thấy mô hình. Bắt đầu huấn luyện chiều VIETNAMESE -> KOREAN...")
    train_single_direction(
        train_df=train_df,
        val_df=val_df,
        src_col="vi",
        tgt_col="ko",
        model_save_path=PATH_VI2KO_MODEL,
        checkpoint_path=PATH_VI2KO_CKPT
    )

Chưa tìm thấy mô hình. Bắt đầu huấn luyện chiều VIETNAMESE -> KOREAN...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]


BẮT ĐẦU TRAIN CHIỀU: VI -> KO (40,000 MẪU)


Step,Training Loss
100,5.986290
200,5.512810
300,5.460289
400,5.417333
500,5.533574
600,5.552103
700,5.509798
800,5.482263
900,5.497015
1000,5.509054


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu mô hình thành công tại: /content/drive/MyDrive/Xử Lý Ngôn Ngữ Tự Nhiên/Tiểu Luận/Dataset/Data-vi-ko/nllb_vi2ko_model


In [ ]:
# Train Mô hình 2 (Hàn -> Việt)
PATH_KO2VI_MODEL = os.path.join(FOLDER_PATH, "nllb_ko2vi_model")
PATH_KO2VI_CKPT = os.path.join(FOLDER_PATH, "checkpoint_ko2vi")

# Kiểm tra xem mô hình đã được lưu đầy đủ trên Drive chưa
config_file = os.path.join(PATH_KO2VI_MODEL, "config.json")

if os.path.exists(config_file):
    print(f"Mô hình Hàn -> Việt ĐÃ TỒN TẠI tại: {PATH_KO2VI_MODEL}")
    print("-> Bỏ qua bước huấn luyện và chuyển sang cell tiếp theo!")
else:
    print("Chưa tìm thấy mô hình. Bắt đầu huấn luyện chiều HÀN -> VIỆT...")
    train_single_direction(
        train_df=train_df,
        val_df=val_df,
        src_col="ko",
        tgt_col="vi",
        model_save_path=PATH_KO2VI_MODEL,
        checkpoint_path=PATH_KO2VI_CKPT
    )

Chưa tìm thấy mô hình. Bắt đầu huấn luyện chiều HÀN -> VIỆT...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]


BẮT ĐẦU TRAIN CHIỀU: KO -> VI (40,000 MẪU)


Step,Training Loss
100,5.118817
200,4.576117
300,4.531665
400,4.476705
500,4.563340
600,4.570289
700,4.537523
800,4.580504
900,4.522297
1000,4.527597


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu mô hình thành công tại: /content/drive/MyDrive/Xử Lý Ngôn Ngữ Tự Nhiên/Tiểu Luận/Dataset/Data-vi-ko/nllb_ko2vi_model


In [ ]:
#Trích xuất Subword Tokens & Embedding thử nghiệm trên tập Test
import random

test_sample = test_df.sample(1, random_state=42).iloc[0]
sample_vi = test_sample['vi']
sample_ko = test_sample['ko']

eval_tokenizer = AutoTokenizer.from_pretrained(PATH_VI2KO_MODEL)

print(" KIỂM TRA SUBWORD TOKENS TRÊN TẬP TEST ")
print("Câu tiếng Việt:", sample_vi)
print("Subwords Việt :", eval_tokenizer.tokenize(sample_vi))
print("Câu tiếng Hàn :", sample_ko)
print("Subwords Hàn  :", eval_tokenizer.tokenize(sample_ko))

 KIỂM TRA SUBWORD TOKENS TRÊN TẬP TEST 
Câu tiếng Việt: Điều này có nghĩa là họ là những người nhẫn tâm và hung hãn nhất trong tất cả các tù nhân.
Subwords Việt : ['▁Điều', '▁này', '▁có', '▁nghĩa', '▁là', '▁họ', '▁là', '▁những', '▁người', '▁nhẫn', '▁tâm', '▁và', '▁hung', '▁hã', 'n', '▁nhất', '▁trong', '▁tất', '▁cả', '▁các', '▁tù', '▁nhân', '.']
Câu tiếng Hàn : 그 말은 제 연구 대상들이 재소자들 중에서도 가장 냉담하고 공격적이란 뜻이었습니다.
Subwords Hàn  : ['▁그', '▁말은', '▁제', '▁연구', '▁대상', '들이', '▁재', '소', '자들', '▁중', '에서도', '▁가장', '▁냉', '담', '하고', '▁공격', '적이', '란', '▁뜻', '이었습니다', '.']


In [ ]:
import torch
import evaluate
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

def evaluate_quick(model_path, test_df, src_col, tgt_col, sample_size=5000):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # 1. Xác định mã ngôn ngữ NLLB
    src_lang = "vie_Latn" if src_col == "vi" else "kor_Hang"
    tgt_lang = "kor_Hang" if src_col == "vi" else "vie_Latn"

    # 2. Tải Tokenizer và Mô hình
    tokenizer = AutoTokenizer.from_pretrained(model_path, src_lang=src_lang, tgt_lang=tgt_lang)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device)
    model.eval()

    #  Lấy ID ngôn ngữ đích chuẩn NLLB
    tgt_lang_id = tokenizer.convert_tokens_to_ids(tgt_lang)

    # Chuẩn bị tập test
    eval_sub = test_df.sample(min(sample_size, len(test_df)), random_state=42).reset_index(drop=True)
    inputs = eval_sub[src_col].astype(str).tolist()
    references = eval_sub[tgt_col].astype(str).tolist()

    # Tải thước đo BLEU & chrF++
    bleu_metric = evaluate.load("bleu")
    chrf_metric = evaluate.load("chrf")

    predictions = []
    batch_size = 32  # Dịch theo batch để chạy 5.000 câu trong ~1-2 phút

    print(f"\n--- Đang đánh giá chiều: {src_col.upper()} -> {tgt_col.upper()} ({len(eval_sub):,} mẫu) ---")

    for i in tqdm(range(0, len(inputs), batch_size)):
        batch_inputs = inputs[i : i + batch_size]
        encoded = tokenizer(
            batch_inputs,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            generated_tokens = model.generate(
                **encoded,
                forced_bos_token_id=tgt_lang_id,
                max_length=128,
                num_beams=2,
                no_repeat_ngram_size=3
            )

        decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        predictions.extend(decoded)

    # Tính BLEU và chrF
    formatted_refs = [[ref] for ref in references]

    bleu_score = bleu_metric.compute(predictions=predictions, references=formatted_refs)["bleu"] * 100
    chrf_score = chrf_metric.compute(predictions=predictions, references=references)["score"]

    print(f"-> Kết quả [{src_col.upper()} -> {tgt_col.upper()}]: BLEU = {bleu_score:.2f} | chrF++ = {chrf_score:.2f}")
    return bleu_score, chrf_score

In [ ]:
test_file_path = os.path.join(FOLDER_PATH, "test.csv")

test_df = pd.read_csv(test_file_path)

print("Đã tải thành công test_df!")
print("Danh sách các cột:", test_df.columns.tolist())
print("Số lượng bản ghi:", len(test_df))


Đã tải thành công test_df!
Danh sách các cột: ['vi', 'ko']
Số lượng bản ghi: 30613


In [ ]:
import gc
import os
import evaluate
import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

FOLDER_PATH = (
    "/content/drive/MyDrive/Xử Lý Ngôn Ngữ Tự Nhiên/Tiểu Luận/Dataset/Data-vi-ko"
)
PATH_VI2KO_MODEL = os.path.join(FOLDER_PATH, "nllb_vi2ko_model")
PATH_KO2VI_MODEL = os.path.join(FOLDER_PATH, "nllb_ko2vi_model")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def evaluate_quick(
    model_path,
    test_df,
    src_lang,
    tgt_lang,
    sample_size=1000,
    output_csv=None,
    batch_size=32,
):
  config_file = os.path.join(model_path, "config.json")
  if not os.path.exists(config_file):
    print(f"Không tìm thấy mô hình tại: {model_path}")
    return None, None, None

  print(f"\nĐang tải mô hình từ: {model_path}")
  tokenizer = AutoTokenizer.from_pretrained(model_path)
  model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device)

  # Cấu hình mã ngôn ngữ chuẩn cho mô hình NLLB
  src_code = "vie_Latn" if src_lang == "vi" else "kor_Hang"
  tgt_code = "kor_Hang" if src_lang == "vi" else "vie_Latn"

  tokenizer.src_lang = src_code
  forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_code)

  # Load các thư viện đo điểm
  sacrebleu = evaluate.load("sacrebleu")
  chrf = evaluate.load("chrf")
  ter = evaluate.load("ter")

  eval_df = (
      test_df.sample(n=min(sample_size, len(test_df)), random_state=42)
      .copy()
      .reset_index(drop=True)
  )

  predictions = []
  references = []
  ref_list_for_ter = []

  print(
      f" Bắt đầu đánh giá chiều {src_lang.upper()} -> {tgt_lang.upper()}"
      f" ({len(eval_df):,} mẫu)..."
  )
  model.eval()

  # Dịch theo dạng Lô (Batch)
  with torch.no_grad():
    for i in range(0, len(eval_df), batch_size):
      batch_rows = eval_df.iloc[i : i + batch_size]

      # NLLB không sử dụng prefix chuỗi dạng text như T5
      src_texts = [str(text) for text in batch_rows[src_lang]]
      tgt_texts = [str(text) for text in batch_rows[tgt_lang]]

      inputs = tokenizer(
          src_texts,
          return_tensors="pt",
          padding=True,
          truncation=True,
          max_length=128,
      ).to(device)

      # Truyền forced_bos_token_id để NLLB biết ngôn ngữ đầu ra
      outputs = model.generate(
          **inputs,
          forced_bos_token_id=forced_bos_token_id,
          max_length=128,
          num_beams=2,
      )
      pred_texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)

      predictions.extend(pred_texts)
      for tgt_text in tgt_texts:
        references.append([tgt_text])
        ref_list_for_ter.append(tgt_text)

  # Tính toán các chỉ số
  bleu_score = sacrebleu.compute(
      predictions=predictions, references=references
  )["score"]
  chrf_score = chrf.compute(predictions=predictions, references=references)[
      "score"
  ]
  ter_score = ter.compute(
      predictions=predictions, references=ref_list_for_ter
  )["score"]

  print(f" KẾT QUẢ {src_lang.upper()} -> {tgt_lang.upper()}:")
  print(f"   - BLEU  : {bleu_score:.2f}")
  print(f"   - chrF++: {chrf_score:.2f}")
  print(f"   - TER   : {ter_score:.2f}")

  # Lưu kết quả ra file CSV
  eval_df["model_prediction"] = predictions
  if output_csv:
    save_directory = (
        "/content/drive/MyDrive"
        if os.path.exists("/content/drive/MyDrive")
        else "."
    )
    save_path = os.path.join(save_directory, output_csv)
    eval_df.to_csv(save_path, index=False, encoding="utf-8-sig")
    print(f"   -> Đã lưu bảng kết quả dịch chi tiết tại: {save_path}")

  # Giải phóng GPU VRAM
  del model
  gc.collect()
  torch.cuda.empty_cache()

  return bleu_score, chrf_score, ter_score


# ==========================================
# THỰC THI ĐÁNH GIÁ (5000 MẪU)
# ==========================================
eval_samples = 5000 if torch.cuda.is_available() else 100

vi2ko_bleu, vi2ko_chrf, vi2ko_ter = evaluate_quick(
    PATH_VI2KO_MODEL,
    test_df,
    "vi",
    "ko",
    sample_size=eval_samples,
    output_csv="eval_results_vi2ko.csv",
)

ko2vi_bleu, ko2vi_chrf, ko2vi_ter = evaluate_quick(
    PATH_KO2VI_MODEL,
    test_df,
    "ko",
    "vi",
    sample_size=eval_samples,
    output_csv="eval_results_ko2vi.csv",
)


Đang tải mô hình từ: /content/drive/MyDrive/Xử Lý Ngôn Ngữ Tự Nhiên/Tiểu Luận/Dataset/Data-vi-ko/nllb_vi2ko_model


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

 Bắt đầu đánh giá chiều VI -> KO (5,000 mẫu)...
 KẾT QUẢ VI -> KO:
   - BLEU  : 5.67
   - chrF++: 21.53
   - TER   : 92.70
   -> Đã lưu bảng kết quả dịch chi tiết tại: /content/drive/MyDrive/eval_results_vi2ko.csv

Đang tải mô hình từ: /content/drive/MyDrive/Xử Lý Ngôn Ngữ Tự Nhiên/Tiểu Luận/Dataset/Data-vi-ko/nllb_ko2vi_model


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

 Bắt đầu đánh giá chiều KO -> VI (5,000 mẫu)...
 KẾT QUẢ KO -> VI:
   - BLEU  : 18.42
   - chrF++: 38.33
   - TER   : 74.77
   -> Đã lưu bảng kết quả dịch chi tiết tại: /content/drive/MyDrive/eval_results_ko2vi.csv


In [ ]:
import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

PATH_VI2KO = "/content/drive/MyDrive/Xử Lý Ngôn Ngữ Tự Nhiên/Tiểu Luận/Dataset/Data-vi-ko/nllb_vi2ko_model"
PATH_KO2VI = "/content/drive/MyDrive/Xử Lý Ngôn Ngữ Tự Nhiên/Tiểu Luận/Dataset/Data-vi-ko/nllb_ko2vi_model"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Đang tải mô hình lên GPU...")
tokenizer_vi2ko = AutoTokenizer.from_pretrained(PATH_VI2KO, src_lang="vie_Latn", tgt_lang="kor_Hang")
model_vi2ko = AutoModelForSeq2SeqLM.from_pretrained(PATH_VI2KO).to(device)
model_vi2ko.eval()

tokenizer_ko2vi = AutoTokenizer.from_pretrained(PATH_KO2VI, src_lang="kor_Hang", tgt_lang="vie_Latn")
model_ko2vi = AutoModelForSeq2SeqLM.from_pretrained(PATH_KO2VI).to(device)
model_ko2vi.eval()
print("Đã tải xong toàn bộ mô hình!")

def translate_text(text, direction):
    if not text or not text.strip():
        return ""

    if direction == "Tiếng Việt -> Tiếng Hàn":
        tokenizer = tokenizer_vi2ko
        model = model_vi2ko
        tgt_lang = "kor_Hang"
    else:
        tokenizer = tokenizer_ko2vi
        model = model_ko2vi
        tgt_lang = "vie_Latn"

    # SỬA LỖI TẠI ĐÂY: Dùng convert_tokens_to_ids chuẩn NLLB
    forced_bos_id = tokenizer.convert_tokens_to_ids(tgt_lang)

    inputs = tokenizer(text, return_tensors="pt", max_length=128, truncation=True).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_id,
            max_length=128,
            num_beams=4,
            no_repeat_ngram_size=3
        )

    translated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translated

# Giao diện Gradio
with gr.Blocks(title="Hệ Thống Dịch Máy Việt - Hàn") as demo:
    gr.Markdown("## Hệ Thống Dịch Máy Song Hướng Việt - Hàn ")

    direction = gr.Radio(
        choices=["Tiếng Việt -> Tiếng Hàn", "Tiếng Hàn -> Tiếng Việt"],
        value="Tiếng Việt -> Tiếng Hàn",
        label="Chọn chiều dịch"
    )

    with gr.Row():
        input_text = gr.Textbox(lines=4, placeholder="Nhập văn bản cần dịch...", label="Văn bản gốc")
        output_text = gr.Textbox(lines=4, label="Kết quả dịch", interactive=False)

    btn = gr.Button("Dịch ngay", variant="primary")
    btn.click(fn=translate_text, inputs=[input_text, direction], outputs=output_text)

demo.queue().launch(share=True, debug=True)

Đang tải mô hình lên GPU...


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

Đã tải xong toàn bộ mô hình!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://118fda4e8b20bc57ad.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://118fda4e8b20bc57ad.gradio.live
